# Tutorial 1 — Detect privileged pods in raw audit logs (20 minutes)

**Start here:** `./setup.sh --notebooks`. Choose Python 3, then **Run → Run All Cells**.
Every example and practice cell runs before you edit anything. Use Shift+Enter to rerun one cell.

Scenario: investigate Kubernetes audit activity on `payments-prod` across EKS, GKE, and AKS.
You will inspect raw events, run a working privileged-pod detection, and modify a pod-exec rule.
No cloud account or model API key is needed. See `WORKSHOP.md` for the full route.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts at the repository root or inside tutorials/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "workshop" / "db.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start JupyterLab from the workshop checkout: ./setup.sh --notebooks")
sys.path.insert(0, str(ROOT))
from workshop.tutorial import open_workshop, q, compare, evaluate

con = open_workshop(raw_only=True)


## 1. Inspect the three source formats
`raw_eks.event` and `raw_aks.event` contain native Kubernetes audit events.
`raw_gke.entry` contains a Cloud Logging envelope with `protoPayload`.
Counts and samples come from your local bundled dataset.


In [ ]:
for cloud in ("eks", "gke", "aks"):
    print(cloud, con.sql(f"SELECT count(*) FROM raw_{cloud}").fetchone()[0])


In [ ]:
q(con, "SELECT event FROM raw_eks LIMIT 1")


In [ ]:
q(con, "SELECT entry FROM raw_gke LIMIT 1")


## 2. Run the same detection in each cloud
These examples check pod creates with the **first regular container** marked privileged.
They identify candidates, including benign platform workloads; a hit does not prove an attack.
EKS/AKS expose `verb` and `objectRef`; GKE encodes the operation in `methodName`.
Read the SQL before running it. Every detection returns `event_id` for evaluation.


### EKS


In [ ]:
PRIVILEGED_EKS = """
SELECT json_extract_string(event, '$.auditID') AS event_id,
       json_extract_string(event, '$.user.username') AS principal,
       json_extract_string(event, '$.objectRef.namespace') AS namespace,
       json_extract_string(event, '$.requestObject.spec.containers[0].image') AS image
FROM raw_eks
WHERE json_extract_string(event, '$.verb') = 'create'
  AND json_extract_string(event, '$.objectRef.resource') = 'pods'
  AND json_extract_string(event, '$.objectRef.subresource') IS NULL
  AND json_extract_string(event, '$.requestObject.spec.containers[0].securityContext.privileged') = 'true'
"""
q(con, PRIVILEGED_EKS)


### AKS


In [ ]:
PRIVILEGED_AKS = """
SELECT json_extract_string(event, '$.auditID') AS event_id,
       json_extract_string(event, '$.user.username') AS principal,
       json_extract_string(event, '$.objectRef.namespace') AS namespace,
       json_extract_string(event, '$.requestObject.spec.containers[0].image') AS image
FROM raw_aks
WHERE json_extract_string(event, '$.verb') = 'create'
  AND json_extract_string(event, '$.objectRef.resource') = 'pods'
  AND json_extract_string(event, '$.objectRef.subresource') IS NULL
  AND json_extract_string(event, '$.requestObject.spec.containers[0].securityContext.privileged') = 'true'
"""
q(con, PRIVILEGED_AKS)


### GKE


In [ ]:
PRIVILEGED_GKE = """
SELECT json_extract_string(entry, '$.operation.id') AS event_id,
       json_extract_string(entry, '$.protoPayload.authenticationInfo.principalEmail') AS principal,
       split_part(json_extract_string(entry, '$.protoPayload.resourceName'), '/', 4) AS namespace,
       json_extract_string(entry, '$.protoPayload.request.spec.containers[0].image') AS image
FROM raw_gke
WHERE json_extract_string(entry, '$.protoPayload.methodName') = 'io.k8s.core.v1.pods.create'
  AND json_extract_string(entry, '$.protoPayload.request.spec.containers[0].securityContext.privileged') = 'true'
"""
q(con, PRIVILEGED_GKE)


In [ ]:
for cloud, sql in {"AWS": PRIVILEGED_EKS, "Azure": PRIVILEGED_AKS, "GCP": PRIVILEGED_GKE}.items():
    print(cloud, con.sql(f"SELECT count(*) FROM ({sql})").fetchone()[0])


## 3. Practice: change an EKS pod-exec detection
The baseline is complete. Copy this cell using **Edit → Duplicate Cell**, or insert a code cell
with the `+` toolbar button. Add namespace, principal, or time filters and inspect the difference.
`json_extract_string` avoids DuckDB's JSON-arrow operator precedence trap.
JSON array indexes start at 0; SQL LIST indexes start at 1.


In [ ]:
POD_EXEC_EKS = """
SELECT json_extract_string(event, '$.auditID') AS event_id,
       json_extract_string(event, '$.user.username') AS principal,
       json_extract_string(event, '$.sourceIPs[0]') AS src_ip,
       json_extract_string(event, '$.objectRef.name') AS pod
FROM raw_eks
WHERE json_extract_string(event, '$.verb') = 'create'
  AND json_extract_string(event, '$.objectRef.resource') = 'pods'
  AND json_extract_string(event, '$.objectRef.subresource') = 'exec'
"""
q(con, POD_EXEC_EKS)


In [ ]:
# YOUR QUERY: edit this string, then rerun this cell and the comparison below.
MY_EXEC_SQL = POD_EXEC_EKS + "\n  AND json_extract_string(event, '$.objectRef.namespace') = 'payments'"
q(con, MY_EXEC_SQL)


In [ ]:
compare(con, MY_EXEC_SQL, POD_EXEC_EKS)


A narrower query will normally show missing IDs relative to the broad baseline. That is useful
feedback, not a notebook failure. Remove your additional filter expression to recover an exact match.
For the full raw exercise, open `labs/lab1_raw/01_raw_detections.ipynb`; its remaining TODO
cells intentionally do not pass the grader until you implement them.

## 4. Evaluate separately from detection
The synthetic ground truth labels outputs **after** the rule runs. Never build a detection
from the ground-truth IDs. Inspect both true hits and benign lookalikes.


In [ ]:
evaluate(con, PRIVILEGED_EKS)


**Checkpoint:** explain why three SQL variants were necessary and why privileged workloads
can be benign. Save with Ctrl+S / Cmd+S. Continue with `02_normalized_detections.ipynb`;
it is independent, so you can close this kernel first to free memory.


In [ ]:
# Add your next query below: q(con, "SELECT ...")
# Leave con open while exploring; shut down this kernel when finished.
